# Standing Shoulder Internal/External Rotation - TCN

Este notebook usa `Standing_Shoulder_Internal_External_Rotation.py` para:
- Entrada: ventana temporal de frames desde JSON (`jsonEjemplo.json`)
- Features biomecanicas (angulo hombro-brazo, altura relativa, elevacion escapular, tronco, simetria, velocidad angular, rango maximo)
- Modelo: TCN
- Salidas: correcta/incorrecta, severidad y errores detectados
- Objetivo principal: maximizar sensibilidad/recall de ejecuciones incorrectas

## 0. Dependencias
Si usas Python 3.14, TensorFlow puede no estar disponible.
Para entrenar TCN, usa preferiblemente Python 3.11 o 3.12.

In [10]:
# Ejecuta solo si necesitas instalar en este kernel
%pip install -q numpy pandas scikit-learn joblib tensorflow

Note: you may need to restart the kernel to use updated packages.


## 1. Importaciones

In [11]:
from pathlib import Path
import json
import importlib
import numpy as np
import joblib

import Standing_Shoulder_Internal_External_Rotation as ssr
ssr = importlib.reload(ssr)

print(f'BASE_DIR: {Path.cwd()}')
print(f'JSON ejemplo: {ssr.JSON_EXAMPLE_PATH}')
print(f'Dataset real esperado: {ssr.DATASET_ROOT}')
print(f'Artifacts: {ssr.ARTIFACTS_DIR}')

BASE_DIR: c:\Users\marco\notebooks\modelo_rehabilitacion\standing_shoulder_internal_external_rotation
JSON ejemplo: C:\Users\marco\notebooks\modelo_rehabilitacion\jsonEjemplo.json
Dataset real esperado: C:\Users\marco\notebooks\modelo_rehabilitacion\dataset
Artifacts: C:\Users\marco\notebooks\modelo_rehabilitacion\standing_shoulder_internal_external_rotation\standing_shoulder_internal_external_rotation_artifacts


## 2. Vista rapida del JSON de entrada

In [12]:
payload = ssr.load_json_payload(ssr.JSON_EXAMPLE_PATH)

print('Claves de primer nivel:', list(payload.keys()))
if 'puntos_clave' in payload:
    print('Joints disponibles en ejemplo:', list(payload['puntos_clave'].keys()))
elif 'frames' in payload:
    print('Cantidad de frames en ejemplo:', len(payload['frames']))
else:
    print('Formato detectado:', type(payload))

Claves de primer nivel: ['ejercicio_id', 'secuencia']
Formato detectado: <class 'dict'>


## 3. Construccion de dataset

Usa tu dataset real UI-PRMD desde la carpeta `dataset`:

- `dataset/Segmented Movements/Segmented Movements/Kinect/Positions`

- `dataset/Incorrect Segmented Movements/Incorrect Segmented Movements/Kinect/Positions`

- `dataset/Movements/Movements/Kinect/Positions`

- `dataset/Incorrect Movements/Incorrect Movements/Kinect/Positions`

El entrenamiento queda forzado a UI-PRMD (m07) y no usa fallback a carpetas JSON/sinteticas.

In [13]:
X, y, meta, dataset_source = ssr.get_dataset()
print(f'dataset_source: {dataset_source}')
print(f'X shape: {X.shape}')
print(f'y balance: {np.bincount(y)}')

coverage = meta.attrs.get('coverage', {})
if coverage:
    print('\nCobertura loader m07:')
    for k, v in coverage.items():
        print(f'  {k}: {v}')

if 'folder_split' in meta.columns:
    print('\nDistribucion por carpeta origen:')
    print(meta.groupby(['folder_split', 'label']).size().unstack(fill_value=0))

meta.head()

dataset_source: ui_prmd_kinect_positions
X shape: (220, 48, 10)
y balance: [110 110]

Cobertura loader m07:
  candidate_files: 220
  used_sequences: 220
  skipped_load: 0
  skipped_shape: 0
  skipped_features: 0

Distribucion por carpeta origen:
label                            0    1
folder_split                           
Incorrect Movements              0   10
Incorrect Segmented Movements    0  100
Movements                       10    0
Segmented Movements            100    0


,sequence_id,subject_id,label,file,folder_split,is_segmented,n_frames_raw,max_shoulder_angle_deg,mean_shoulder_angle_deg,trunk_tilt_p95_deg,shoulder_elevation_p95,symmetry_p95_deg,velocity_std_deg,wrist_above_elbow_mean,valid_points_ratio_mean
0,m07_s01_e01_positions,1,0,m07_s01_e01_positions.txt,Segmented Movements,True,63,97.681976,97.096092,173.548477,0.027408,77.096939,0.083988,0.006749,1.0
1,m07_s01_e02_positions,1,0,m07_s01_e02_positions.txt,Segmented Movements,True,66,98.432243,97.368744,173.701950,0.042471,77.403893,0.110933,0.005806,1.0
2,m07_s01_e03_positions,1,0,m07_s01_e03_positions.txt,Segmented Movements,True,64,98.386955,97.422112,173.294083,0.007128,76.588165,0.104428,0.006855,1.0
3,m07_s01_e04_positions,1,0,m07_s01_e04_positions.txt,Segmented Movements,True,63,100.261536,99.255653,171.559479,0.014519,73.118973,0.114015,0.006307,1.0
4,m07_s01_e05_positions,1,0,m07_s01_e05_positions.txt,Segmented Movements,True,62,98.994759,98.210144,172.471634,0.054945,74.943268,0.095097,0.003836,1.0


## 4. Entrenamiento del TCN (foco en recall)
Esta celda entrena, ajusta umbral para recall objetivo y exporta artifacts.

In [14]:
results = None
try:
    results = ssr.train_and_export()
    print('Entrenamiento completado.')
    print('Metricas principales:')
    for k, v in results['metrics'].items():
        print(f'  {k}: {v}')
except RuntimeError as exc:
    print(str(exc))
    print('Sugerencia: usa un entorno Python 3.11/3.12 para TensorFlow y vuelve a ejecutar.')

=== Standing Shoulder Internal/External Rotation / TCN ===
Dataset usado        : ui_prmd_kinect_positions
Muestras totales     : 220
Threshold optimizado : 0.4775
Recall incorrecto    : 0.9630
Precision incorrecto : 0.6500
F1 incorrecto        : 0.7761
ROC-AUC              : 0.8413
Matriz de confusión:
[[14 14]
 [ 1 26]]
Reporte de clasificación:
              precision    recall  f1-score   support

    correcta     0.9333    0.5000    0.6512        28
  incorrecta     0.6500    0.9630    0.7761        27

    accuracy                         0.7273        55
   macro avg     0.7917    0.7315    0.7136        55
weighted avg     0.7942    0.7273    0.7125        55

Demo de inferencia:
{
  "exercise": "Standing Shoulder Internal/External Rotation",
  "movement_id": 7,
  "label": 1,
  "classification": "incorrecta",
  "probability_incorrect": 1.0,
  "threshold": 0.4775,
  "severity": "severa",
  "errors_detected": [
    "codo_separado_del_torso",
    "inclinacion_excesiva_tronco",
   

## 5. Inferencia demo
Salida esperada:
- etiqueta: correcta / incorrecta
- severidad
- errores_detectados

In [15]:
try:
    tf = ssr.require_tensorflow()
    bundle_path = ssr.ARTIFACTS_DIR / 'standing_shoulder_internal_external_rotation_bundle.joblib'
    model_path = ssr.ARTIFACTS_DIR / 'standing_shoulder_internal_external_rotation_tcn.keras'

    if not bundle_path.exists() or not model_path.exists():
        raise FileNotFoundError('No existen artifacts aun. Ejecuta la celda de entrenamiento primero.')

    bundle = joblib.load(bundle_path)
    model = tf.keras.models.load_model(model_path)

    demo_payload = ssr.load_json_payload(ssr.JSON_EXAMPLE_PATH)
    demo_frames, _ = ssr.generate_temporal_window_from_example(
        payload=demo_payload,
        rng=np.random.default_rng(123),
        incorrect=True,
        n_frames=ssr.WINDOW_SIZE,
    )

    pred = ssr.infer_from_window(demo_frames, model=model, bundle=bundle)
    print(json.dumps(pred, ensure_ascii=False, indent=2))
except Exception as exc:
    print(f'No se pudo ejecutar inferencia: {exc}')

{
  "exercise": "Standing Shoulder Internal/External Rotation",
  "movement_id": 7,
  "label": 1,
  "classification": "incorrecta",
  "probability_incorrect": 1.0,
  "threshold": 0.4775,
  "severity": "severa",
  "errors_detected": [
    "codo_separado_del_torso",
    "inclinacion_excesiva_tronco",
    "elevacion_escapular_excesiva",
    "velocidad_angular_irregular"
  ],
  "biomechanical_summary": {
    "max_shoulder_angle_deg": 179.5813,
    "mean_shoulder_angle_deg": 153.6577,
    "trunk_tilt_p95_deg": 148.983,
    "shoulder_elevation_p95": 0.3564,
    "symmetry_p95_deg": 17.9175,
    "velocity_std_deg": 5.0483,
    "wrist_above_elbow_mean": 0.3249,
    "valid_points_ratio_mean": 1.0
  }
}


## 6. Uso en produccion (resumen)

1. Carga artifacts exportados (`.joblib` + `.keras`).
2. Recibe payload JSON con `frames` o `puntos_clave`.
3. Normaliza a secuencia con `ssr.coerce_frames(...)`.
4. Llama `ssr.infer_from_window(...)` y devuelve:
   - `etiqueta` (`correcta` / `incorrecta`)
   - `severidad`
   - `errores_detectados`
   - `probabilidad_incorrecta`
   - `resumen_biomecanico`

Ejemplo minimo de carga:

```python
from pathlib import Path
import joblib
import tensorflow as tf
import Standing_Shoulder_Internal_External_Rotation as ssr

artifacts = Path('standing_shoulder_internal_external_rotation_artifacts')
bundle = joblib.load(artifacts / 'standing_shoulder_internal_external_rotation_bundle.joblib')
model = tf.keras.models.load_model(artifacts / 'standing_shoulder_internal_external_rotation_tcn.keras')
```

Ejemplo minimo de prediccion:

```python
def predecir_rotacion(payload_json: dict) -> dict:
    frames = ssr.coerce_frames(payload_json)
    if len(frames) < 2:
        frames = frames * max(2, ssr.WINDOW_SIZE)
    return ssr.infer_from_window(frames_json=frames, model=model, bundle=bundle)
```

Nota: para entrenar TCN usa Python 3.11 o 3.12.

In [16]:
from pathlib import Path
import joblib
import tensorflow as tf
import Standing_Shoulder_Internal_External_Rotation as ssr

artifacts = Path('standing_shoulder_internal_external_rotation_artifacts')
bundle = joblib.load(artifacts / 'standing_shoulder_internal_external_rotation_bundle.joblib')
model = tf.keras.models.load_model(artifacts / 'standing_shoulder_internal_external_rotation_tcn.keras')

def predecir_rotacion(payload_json: dict) -> dict:
    frames = ssr.coerce_frames(payload_json)
    if len(frames) < 2:
        frames = frames * max(2, ssr.WINDOW_SIZE)
    return ssr.infer_from_window(frames_json=frames, model=model, bundle=bundle)




In [17]:
predecir_rotacion({ "metadatos": { "dispositivo": "Samsung Tab S3", "orientacion": "Portrait", "pantalla_px": {
"ancho": 1536, "alto": 2048 }, "timestamp_ms": 1710584200123 }, "puntos_clave": { "nariz":
{"x": 0.50, "y": 0.20, "z": -0.15}, "hombro_d": {"x": 0.40, "y": 0.35, "z": -0.05}, "codo_d": {"x":
0.35, "y": 0.50, "z": -0.08}, "muÃ±eca_d": {"x": 0.30, "y": 0.65, "z": -0.12} } })

{'exercise': 'Standing Shoulder Internal/External Rotation',
 'movement_id': 7,
 'label': 1,
 'classification': 'incorrecta',
 'probability_incorrect': 0.5007,
 'threshold': 0.4775,
 'severity': 'severa',
 'errors_detected': ['codo_separado_del_torso', 'inclinacion_excesiva_tronco'],
 'biomechanical_summary': {'max_shoulder_angle_deg': 164.7449,
  'mean_shoulder_angle_deg': 164.7449,
  'trunk_tilt_p95_deg': 33.6901,
  'shoulder_elevation_p95': 0.0,
  'symmetry_p95_deg': 0.0,
  'velocity_std_deg': 0.0,
  'wrist_above_elbow_mean': nan,
  'valid_points_ratio_mean': 0.5}}